# 10 · Coembedding

`rx.coembed(reactant, product)` builds reactant and product conformers in one frame, as start points for a reaction
path or a transition-state search. The product comes back on the reactant's atom order, the atoms that do not react are
aligned, and the reacting atoms move between the two. Each render draws the first pair's product over its reactant.

In [ ]:
import numpy as np
import rxembed as rx
import xyzrender

## An SN2 alkylation

Methyl iodide alkylates the ring N of an aryl boronate: the C-N bond forms as the C-I bond breaks. Atom maps name every
atom in both SMILES, so the product's atoms are put on the reactant's order, and the boronate and ring framework align
while the methyl moves from I to N.

In [ ]:
SN2_REACTANT = "[C:1]([I:2])([H:3])([H:4])[H:5].[C:6]([c:7]1[c:8]([B:9]2[O:10][C:11]([C:12]([H:29])([H:30])[H:31])([C:13]([H:32])([H:33])[H:34])[C:14]([C:15]([H:35])([H:36])[H:37])([C:16]([H:38])([H:39])[H:40])[O:17]2)[c:18]([H:41])[c:19]([H:42])[c:20]2[c:21]1[N:22]([H:43])[C:23]([H:44])([H:45])[C:24]([H:46])([H:47])[O:25]2)([H:26])([H:27])[H:28]"
SN2_PRODUCT = "[C:1]([H:3])([H:4])([H:5])[N@+:22]1([H:43])[c:21]2[c:7]([C:6]([H:26])([H:27])[H:28])[c:8]([B:9]3[O:10][C:11]([C:12]([H:29])([H:30])[H:31])([C:13]([H:32])([H:33])[H:34])[C:14]([C:15]([H:35])([H:36])[H:37])([C:16]([H:38])([H:39])[H:40])[O:17]3)[c:18]([H:41])[c:19]([H:42])[c:20]2[O:25][C:24]([H:46])([H:47])[C:23]1([H:44])[H:45].[I-:2]"
reactant, product = (rx.parse_smiles(text, remove_hs=False) for text in (SN2_REACTANT, SN2_PRODUCT))
sn2 = rx.coembed(reactant, product, n=2, seed=1)
for cid in sn2.ids:
    print(f"pair {cid}: spectators {sn2.alignment_scores[cid]:.2f} A, reacting atoms {sn2.react_rmsds[cid]:.2f} A")
pair = sn2[0]
xyzrender.render(xyzrender.load(pair.reactant.mol), overlay=xyzrender.load(pair.product.mol), auto_align=False, hy=True)

## A Diels-Alder pair

Butadiene and ethylene close a cyclohexene ring as two C-C bonds form. `rx.changes` reads the bonds each endpoint's
coordinates make and finds none formed or broken, so both endpoints are the species they were written as; the diene's
C1-C2-C3-C4 dihedral shows whether the reactant's diene is s-cis, ready to close.

In [ ]:
DIELS_ALDER = "[CH2:1]=[CH:2][CH:3]=[CH2:4].[CH2:5]=[CH2:6]>>[CH2:1]1[CH:2]=[CH:3][CH2:4][CH2:5][CH2:6]1"
diels_alder = rx.coembed(DIELS_ALDER, n=4, seed=1)
for k, cid in enumerate(diels_alder.ids):
    pair = diels_alder[k]
    print(f"pair {cid}: diene {pair.reactant.measure((0, 1, 2, 3))['mean']:.0f} deg")
    print("  reactant", rx.changes(diels_alder.reactant.mol, cid))
    print("  product ", rx.changes(diels_alder.product.mol, cid))
pair = diels_alder[0]
xyzrender.render(xyzrender.load(pair.reactant.mol), overlay=xyzrender.load(pair.product.mol), auto_align=False, hy=True)

## An atropisomer turn

Reactant and product are one molecule with opposite hands of its N-aryl axis, written as CX `wU` wedges on different
bonds. Only the stereo changes, so the aryl ring turns between the endpoints while the rest aligns, and each endpoint
keeps its own hand.

In [ ]:
ATROP_REACTANT = "[N:1](=[C:2]1/[S:3][C@@:4]([H:20])([C:21]([F:22])([F:23])[F:24])[C:5]([H:18])([H:19])[C:13](=[O:14])[N:6]1[c:7]1[c:8]([C:26]([H:27])([C:28]([H:29])([H:30])[H:31])[C:32]([H:33])([H:34])[H:35])[c:9]([H:15])[c:10]([H:25])[c:11]([H:16])[c:12]1[H:17])/[S:36](=[O:37])(=[O:38])[c:39]1[c:40]([H:43])[c:42]([H:47])[c:46]([H:49])[c:44]([H:48])[c:41]1[H:45] |wU:14.14|"
ATROP_PRODUCT = "[N:1](=[C:2]1/[S:3][C@@:4]([H:20])([C:21]([F:22])([F:23])[F:24])[C:5]([H:18])([H:19])[C:13](=[O:14])[N:6]1[c:7]1[c:8]([C:26]([H:27])([C:28]([H:29])([H:30])[H:31])[C:32]([H:33])([H:34])[H:35])[c:9]([H:15])[c:10]([H:25])[c:11]([H:16])[c:12]1[H:17])/[S:36](=[O:37])(=[O:38])[c:39]1[c:40]([H:43])[c:42]([H:47])[c:46]([H:49])[c:44]([H:48])[c:41]1[H:45] |wU:14.13|"
reactant, product = (rx.parse_smiles(text, remove_hs=False) for text in (ATROP_REACTANT, ATROP_PRODUCT))
atrop = rx.coembed(reactant, product, n=2, seed=1)
print("reactant", rx.identity(atrop.reactant.mol).stereo, "| product", rx.identity(atrop.product.mol).stereo)
pair = atrop[0]
xyzrender.render(xyzrender.load(pair.reactant.mol), overlay=xyzrender.load(pair.product.mol), auto_align=False, hy=True)

## A catalyst grip held in both endpoints

The BIMP catalyst of [06_catalysis_with_nci](06_catalysis_with_nci.ipynb) holds its substrate as a C-C bond breaks
and an O-C bond forms in its place. `contacts=` holds the first H-bond mode `rx.contacts` proposes for the reactant in
both endpoints, and `binding_modes()` counts the contact kinds each endpoint makes after relaxation. Each
endpoint is drawn alone with the contacts xyzgraph detects dotted.

In [ ]:
BIMP_REACTANT = "[H:84][N:85]([C:86]([N:88]([C@@:92]([C:98]([N+:104]([H:117])=[P:118]([c:128]1[c:131]([H:138])[c:137]([H:150])[c:149]([O:158][C:161]([H:164])([H:165])[H:166])[c:139]([H:151])[c:132]1[H:140])([c:129]1[c:133]([H:142])[c:141]([H:153])[c:152]([O:159][C:162]([H:167])([H:168])[H:169])[c:143]([H:154])[c:134]1[H:144])[c:130]1[c:135]([H:146])[c:145]([H:156])[c:155]([O:160][C:163]([H:170])([H:171])[H:172])[c:147]([H:157])[c:136]1[H:148])([H:105])[H:106])([H:99])[C:100]([C:107]([H:119])([H:120])[H:121])([C:108]([H:122])([H:123])[H:124])[C:109]([H:125])([H:126])[H:127])[H:93])=[S:89])[c:87]1[c:90]([H:95])[c:94]([C:101]([F:110])([F:111])[F:112])[c:102]([H:113])[c:96]([C:103]([F:114])([F:115])[F:116])[c:91]1[H:97].[O:1]=[C:2]1[N:3]([C:4]([c:37]2[c:38]([H:41])[c:40]([H:45])[c:44]([H:47])[c:42]([H:46])[c:39]2[H:43])([c:48]2[c:49]([H:52])[c:51]([H:56])[c:55]([H:58])[c:53]([H:57])[c:50]2[H:54])[c:59]2[c:60]([H:63])[c:62]([H:67])[c:66]([H:69])[c:64]([H:68])[c:61]2[H:65])[c:5]2[c:6]([H:22])[c:7]([H:23])[c:8]([H:24])[c:9]([H:25])[c:10]2[C@@:11]1([O-:12])[C@@:15]1([H:33])[C:14](=[C:13]([H:30])[H:31])[N:32]([S:34](=[O:35])(=[O:36])[c:70]2[c:71]([H:74])[c:73]([H:78])[c:77]([C:80]([H:81])([H:82])[H:83])[c:75]([H:79])[c:72]2[H:76])[c:17]2[c:16]1[c:21]([H:29])[c:20]([H:28])[c:19]([H:27])[c:18]2[H:26]"
BIMP_PRODUCT = "[H:84][N:85]([C:86]([N:88]([C@@:92]([C:98]([N+:104]([H:117])=[P:118]([c:128]1[c:131]([H:138])[c:137]([H:150])[c:149]([O:158][C:161]([H:164])([H:165])[H:166])[c:139]([H:151])[c:132]1[H:140])([c:129]1[c:133]([H:142])[c:141]([H:153])[c:152]([O:159][C:162]([H:167])([H:168])[H:169])[c:143]([H:154])[c:134]1[H:144])[c:130]1[c:135]([H:146])[c:145]([H:156])[c:155]([O:160][C:163]([H:170])([H:171])[H:172])[c:147]([H:157])[c:136]1[H:148])([H:105])[H:106])([H:99])[C:100]([C:107]([H:119])([H:120])[H:121])([C:108]([H:122])([H:123])[H:124])[C:109]([H:125])([H:126])[H:127])[H:93])=[S:89])[c:87]1[c:90]([H:95])[c:94]([C:101]([F:110])([F:111])[F:112])[c:102]([H:113])[c:96]([C:103]([F:114])([F:115])[F:116])[c:91]1[H:97].[O:1]=[c:2]1[n:3]([C:4]([c:37]2[c:38]([H:41])[c:40]([H:45])[c:44]([H:47])[c:42]([H:46])[c:39]2[H:43])([c:48]2[c:49]([H:52])[c:51]([H:56])[c:55]([H:58])[c:53]([H:57])[c:50]2[H:54])[c:59]2[c:60]([H:63])[c:62]([H:67])[c:66]([H:69])[c:64]([H:68])[c:61]2[H:65])[c:5]2[c:6]([H:22])[c:7]([H:23])[c:8]([H:24])[c:9]([H:25])[c:10]2[c-:11]1[O:12][C:13]([c:14]1[c:15]([H:33])[c:16]2[c:17]([c:18]([H:26])[c:19]([H:27])[c:20]([H:28])[c:21]2[H:29])[n:32]1[S:34](=[O:35])(=[O:36])[c:70]1[c:71]([H:74])[c:73]([H:78])[c:77]([C:80]([H:81])([H:82])[H:83])[c:75]([H:79])[c:72]1[H:76])([H:30])[H:31]"
reactant, product = (rx.parse_smiles(text, remove_hs=False) for text in (BIMP_REACTANT, BIMP_PRODUCT))
label, grip = next(iter(rx.contacts(reactant, kinds=("HB",)).items()))
print(label)
bimp = rx.coembed(reactant, product, contacts=grip, n=1, seed=1)
print("reactant", dict(bimp.reactant.binding_modes()), "| product", dict(bimp.product.binding_modes()))
for endpoint in (bimp.reactant, bimp.product):
    display(xyzrender.render(xyzrender.load(endpoint.mol, nci_detect=True)))

## Around a fixed endpoint

`fixed="reactant"` keeps a conformer you already have exactly where it is and builds its partner in the same frame;
here acetone's enol around one acetone conformer.

In [ ]:
acetone = rx.embed("CC(=O)C", n=1, seed=1).minimize().mol
keto_enol = rx.coembed(acetone, "C=C(O)C", fixed="reactant")
unmoved = np.array_equal(keto_enol.reactant.mol.GetConformer().GetPositions(), acetone.GetConformer().GetPositions())
print("acetone unmoved:", unmoved)
xyzrender.render(xyzrender.load(acetone), overlay=xyzrender.load(keto_enol.product.mol), auto_align=False, hy=True)